# Day 58: Lab - Customer Segmentation Mini-Project

## Project Goal
Customer segmentation is one of the most common real-world applications of unsupervised learning: instead of predicting a known label, you're trying to discover natural groups of customers (e.g. high-value loyal spenders vs. occasional bargain shoppers) so marketing, pricing, or product teams can target each group differently. This lab pulls together several techniques from the past few days into one workflow: scale the raw features so clustering treats them fairly, use K-Means with the elbow method to choose a sensible number of segments, then use PCA purely to visualize the resulting segments in 2D. The end goal isn't just "run KMeans" — it's producing segments you can actually interpret and act on, which is why the last step looks at each segment's average feature values rather than stopping at the cluster labels themselves.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 300
# Simulate a customer dataset with 3 behavioral features on very different scales:
# income in tens of thousands, spending score 1-100, purchase frequency 1-50
df = pd.DataFrame({
    'annual_income': np.random.randint(15000, 150000, n),
    'spending_score': np.random.randint(1, 100, n),
    'purchase_frequency': np.random.randint(1, 50, n)
})
print(df.describe())

## Step 1: Scale the Features
K-Means (and most clustering algorithms) group points based on distance, and distance calculations are dominated by whichever feature has the largest numeric range. Here `annual_income` ranges into the hundreds of thousands while `spending_score` only goes up to 100 — without scaling, income alone would essentially decide the clusters and the other two features would barely matter. `StandardScaler` fixes this by transforming each feature to have mean 0 and standard deviation 1, so all three features contribute comparably to the distance calculations that drive clustering.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Scale all 3 features to mean 0, std 1 so no single feature dominates distance-based clustering
X_scaled = StandardScaler().fit_transform(df)

## Step 2: Find k with the Elbow Method
K-Means requires you to choose the number of clusters `k` upfront, and the elbow method is a common heuristic for picking a reasonable value. "Inertia" is the sum of squared distances from each point to its assigned cluster center — it always decreases as `k` increases (more clusters can always fit the data at least as well), so you can't just pick the `k` with lowest inertia or you'd end up with one cluster per point. Instead, you plot inertia against `k` and look for the "elbow" — the point where adding more clusters stops reducing inertia by much, meaning extra clusters are no longer buying you meaningfully tighter groupings. `n_init=10` reruns K-Means 10 times with different random centroid starts and keeps the best result, since K-Means can converge to a poor local optimum depending on its starting point.

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Fit KMeans for k = 1..7 and record inertia (within-cluster sum of squared distances) each time;
# n_init=10 reruns with different centroid initializations and keeps the best fit
inertias = [KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled).inertia_ for k in range(1, 8)]
plt.plot(range(1, 8), inertias, marker='o')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.show()

## Step 3: Cluster & Visualize with PCA
Based on the elbow plot, we settle on `k=4` segments and fit the final K-Means model on the scaled features. Since we can't directly visualize a 3-feature clustering, PCA reduces the scaled data to its top 2 principal components purely so we can plot it — the clustering itself was already done in the original 3D feature space, PCA is only used afterward for the picture. In the scatter plot, each color is a customer segment; well-separated colored regions suggest the clusters are meaningfully distinct, while heavy overlap would suggest the segments blend into each other. Finally, `df.groupby('segment').mean()` translates each cluster back into human-readable terms — the average income, spending score, and purchase frequency per segment — which is what actually makes the segmentation useful to a business (e.g. "segment 2 = high income, low frequency" vs. just "cluster 2").

In [ ]:
from sklearn.decomposition import PCA

# Chosen from the elbow plot above: 4 segments
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['segment'] = kmeans.fit_predict(X_scaled)

# PCA here is only for visualization -- clustering already happened on the full scaled features
X_pca = PCA(n_components=2).fit_transform(X_scaled)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df['segment'], cmap='Set1')
plt.title('Customer Segments (PCA view)')
plt.show()
# Average feature values per segment -- this is what makes the clusters interpretable/actionable
print(df.groupby('segment').mean(numeric_only=True))